In [1]:
# Setup (Imports, API Key, Prompts)
import os
import json
import re
from openai import OpenAI
from tqdm.notebook import tqdm  # ใช้ tqdm.notebook สำหรับ .ipynb
import pandas as pd
import time
from collections import defaultdict
import getpass  # << เพิ่มบรรทัดนี้

print("🚀 Initializing AI Evaluator...")

# --- 1. Setup OpenAI Client ---
# ถ้ายังไม่มี OPENAI_API_KEY ใน env ให้ถามจากผู้ใช้แบบไม่โชว์ตัวอักษร
if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"].startswith("sk-"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"].startswith("sk-"):
    print("=" * 50)
    print("Error: OPENAI_API_KEY not set correctly.")
    print("=" * 50)
else:
    print("OpenAI API Key loaded.")

client = OpenAI()  # จะอ่าน key จาก env โดยอัตโนมัติ
EVALUATION_MODEL = "gpt-4o" # ใช้ "gpt-4o" ตัวเต็มเพื่อการประเมินที่แม่นยำ
print(f"Evaluation Model set to: {EVALUATION_MODEL}")

# --- 2. Define File Paths (Relative to this notebook) ---
OUR_MODEL_FILE = "../llms_therapist/therapist_simulation_results.jsonl"
BASELINE_MODEL_FILE = "../llms_therapist_base/therapist_simulation_results_BASELINE.jsonl"
print(f"Our Model results path: {OUR_MODEL_FILE}")
print(f"Baseline Model results path: {BASELINE_MODEL_FILE}")

# --- 3. Define Evaluation Prompts (Exactly as user provided) ---
SYSTEM_PROMPT = """You are an expert psychological evaluator specializing in Cognitive Behavioral Therapy (CBT) and therapeutic alliance. Your task is to analyze a full counseling session transcript and score the therapist's performance based on the evaluation frameworks used in the MIRROR paper (arXiv:2504.13211v2).

CRITICAL EVALUATION FOCUS:
- You must assess the therapist's ability to perceive "Emotional Subtext"—the underlying feelings that may not be explicitly stated in the client's words but are hinted at through context, tone, or provided emotional metadata.
- A high-performing therapist should move beyond surface-level reflection and identify "Latent Concerns" (e.g., hidden guilt, worthiness issues, or conflicting emotions) to facilitate deeper discovery.

You must evaluate the entire session as a whole, not just individual turns.
"""

USER_PROMPT = """Please evaluate this therapy session. Pay close attention to how the therapist navigates the gap between what the client SAYS and what the client FEELS

[Session Transcript]
{CONVERSATION_TEXT_GOES_HERE}
[/Session Transcript]

[Evaluation Task]
Analyze the therapist's responses. You must be extremely critical and look for "Therapeutic Depth."
**Special Note for Metadata:** Some sessions include metadata (valence, arousal, dissonance flags). These represent the client's non-verbal emotional cues detected by the system. Evaluate how effectively the therapist uses these cues to uncover the client's underlying emotions.

1. **Therapist Skills Assessment (General Counseling):**
 * **Understanding (Score 0-6):** How well did the therapist interpret the client's concerns and feelings? (0 = Not at all, 6 = Perfectly)
 * **Interpersonal Effectiveness (Score 0-6):** How well did the therapist maintain a supportive and therapeutic relationship? (0 = Not at all, 6 = Perfectly)

2. **Client Alliance Assessment:**
 * **Affective Bond (Score 1-5):** How well did the therapist foster an emotional connection, trust, and empathy? (1 = Very Poor, 5 = Very Strong)

3. **CTRS-Based Assessment (0-6 each, use integer scores):**
 * **Collaboration (0-6):** [criteria...]
 * **Guided Discovery (0-6):** [criteria...]
 * **Focus (0-6):** [criteria...]
 * **Strategy (0-6):** [criteria...]

[Output Format]
You MUST return the response in this EXACT JSON structure. Do not skip any fields.

{{
  "therapist_skills": {{
    "understanding": 0.0,
    "interpersonal_effectiveness": 0.0
  }},
  "client_alliance": {{
    "affective_bond": 0.0
  }},
  "ctrs": {{
    "collaboration": 0,
    "guided_discovery": 0,
    "focus": 0,
    "strategy": 0
  }},
  "reasoning": "Explain based on SPECIFIC TURN NUMBERS why this score was given. Highlight where the therapist missed or caught subtext.",
  "comparative_advantage": "Explain why this model is better or worse than a basic text-only therapist. If metadata was provided, did the therapist use it effectively?"
}}
"""
print("✅ Setup complete. Prompts and paths are defined.")

🚀 Initializing AI Evaluator...
OpenAI API Key loaded.
Evaluation Model set to: gpt-4o
Our Model results path: ../llms_therapist/therapist_simulation_results.jsonl
Baseline Model results path: ../llms_therapist_base/therapist_simulation_results_BASELINE.jsonl
✅ Setup complete. Prompts and paths are defined.


In [2]:
# Data Loading & Formatting Functions

import json
from pathlib import Path

BASE = Path(r"C:\Luna-AI-Therapist\dissonance\craft_dialogue")

def load_full_dialogue_jsonl(path: Path) -> str:
    """
    อ่านไฟล์ dialogue_X_full_...jsonl (หนึ่งไฟล์ = 1 dialogue, 1 บรรทัด = 1 turn)
    แล้วแปลงเป็นข้อความแบบ:

    Client: ...
    Therapist: ...
    """
    lines = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            turn = json.loads(line)
            # ปรับ key ตามที่คุณใช้ใน run_dialogue_* (client / therapist)
            lines.append(f"Client: {turn['client']}")
            lines.append(f"Therapist: {turn['therapist']}")
    return "\n".join(lines)

# ชี้ไปที่ไฟล์ทั้งสามชุด
dialogue_files = [
    BASE / "baseline" / "dialogue_1_full_baseline.jsonl",
    BASE / "emotion" / "dialogue_1_full_emotion_online.jsonl",      # ถ้าชื่อไฟล์จริงมี -2 ให้แก้ตรงนี้
    BASE / "dissonance" / "dialogue_1_full_dissonance_online.jsonl",   # ถ้าชื่อไฟล์จริงมี -3 ให้แก้ตรงนี้
]

for p in dialogue_files:
    print(p, p.exists())

def format_conversation_text(turns_list, is_baseline=False):
    """
    Converts a list of turns into a single 'CLIENT: ... THERAPIST: ...' string.
    """
    full_text = ""
    # key สำหรับเคสเก่า (ai_evaluation เดิม)
    therapist_key_old = "therapist_response_baseline" if is_baseline else "therapist_response"

    for turn in turns_list:

        # ฝั่ง client: รองรับทั้ง transcript (ไฟล์เก่า) และ client (ไฟล์ใหม่)
        client_text = turn.get("transcript")
        if client_text is None:
            client_text = turn.get("client", "[missing transcript]")

        # ฝั่ง therapist: รองรับทั้ง therapist_response* (ไฟล์เก่า) และ therapist (ไฟล์ใหม่)
        therapist_text = turn.get(therapist_key_old)
        if therapist_text is None:
            therapist_text = turn.get("therapist", "[missing response]")
        full_text += f"CLIENT: {client_text}\n\n"
        full_text += f"THERAPIST: {therapist_text}\n\n"

    return full_text.strip()

print("✅ Data loading and formatting functions are defined.")

C:\Luna-AI-Therapist\dissonance\craft_dialogue\baseline\dialogue_1_full_baseline.jsonl True
C:\Luna-AI-Therapist\dissonance\craft_dialogue\emotion\dialogue_1_full_emotion_online.jsonl True
C:\Luna-AI-Therapist\dissonance\craft_dialogue\dissonance\dialogue_1_full_dissonance_online.jsonl True
✅ Data loading and formatting functions are defined.


In [3]:
# Evaluation Function (Calling GPT-4o)

def evaluate_session(session_text, max_retries=3):
    """
    Calls the GPT-4o API to get evaluation scores.
    Retries on failure.
    """
    formatted_user_prompt = USER_PROMPT.format(CONVERSATION_TEXT_GOES_HERE=session_text)
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": formatted_user_prompt}
    ]
    
    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=EVALUATION_MODEL,
                messages=messages,
                temperature=0.0, # Crucial for objective scoring
                response_format={"type": "json_object"} # Force JSON output
            )
            
            response_content = completion.choices[0].message.content
            # Try to parse the JSON to ensure it's valid
            scores = json.loads(response_content)
            return scores # Success!
        
        except Exception as e:
            print(f"Warning: Attempt {attempt + 1}/{max_retries} failed. Error: {e}")
            if attempt < max_retries - 1:
                time.sleep(5) # Wait 5 seconds before retrying
            else:
                return {
                    "error": f"Failed after {max_retries} retries.",
                    "last_error": str(e)
                }
    return None # Should not be reached

print("✅ AI Evaluation function is defined.")

✅ AI Evaluation function is defined.


In [4]:
# Main evaluation loop

print("--- Starting Evaluation Process (3 methods, 3 files) ---")

# 1) ชี้ path ไปที่ไฟล์ใหม่ของแต่ละ method
BASE = Path(r"C:\Luna-AI-Therapist\dissonance\craft_dialogue")

METHOD_FILES = {
    "baseline":   BASE / "baseline"   / "dialogue_1_full_baseline.jsonl",
    "emotion":    BASE / "emotion"    / "dialogue_1_full_emotion_online.jsonl",
    "dissonance": BASE / "dissonance" / "dialogue_1_full_dissonance_online.jsonl",
}

def load_single_dialogue_jsonl(path: Path, is_baseline: bool) -> dict:
    """อ่านไฟล์ jsonl 1 ไฟล์ = 1 dialogue แล้วคืน dict {1: [turns...] } ให้ใช้ format_conversation_text เดิมได้."""
    if not path.exists():
        print(f"❌ ERROR: File not found: {path}")
        return {}

    turns = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            # map ให้เหมือนโครงเดิมของ ai_evaluation
            turns.append({
                "transcript": rec["client"],
                ("therapist_response_baseline" if is_baseline else "therapist_response"): rec["therapist"],
            })
    # ใช้ dialogue_id คงที่ = 1
    return {1: turns}

# 2) โหลดทั้ง 3 methods เป็น dict แบบเดิม
baseline_dialogues   = load_single_dialogue_jsonl(METHOD_FILES["baseline"],   is_baseline=True)
emotion_dialogues    = load_single_dialogue_jsonl(METHOD_FILES["emotion"],    is_baseline=False)
dissonance_dialogues = load_single_dialogue_jsonl(METHOD_FILES["dissonance"], is_baseline=False)

# lists สำหรับเก็บผล
baseline_eval_scores   = []
emotion_eval_scores    = []
dissonance_eval_scores = []

# ตอนนี้แต่ละ method มี dialogue_id แค่ {1}
dialogue_ids = [1]

if not baseline_dialogues or not emotion_dialogues or not dissonance_dialogues:
    print("❌ ERROR: Cannot start evaluation. One or more input files failed to load.")
else:
    print("Loaded 1 dialogue per method (baseline / emotion / dissonance).")

    for dialogue_id in tqdm(dialogue_ids, desc="Evaluating Dialogues"):

        # --- Baseline ---
        if dialogue_id in baseline_dialogues:
            print(f"Evaluating BASELINE Dialogue {dialogue_id}...")
            session_text = format_conversation_text(baseline_dialogues[dialogue_id], is_baseline=True)
            scores = evaluate_session(session_text)
            scores["dialogue_id"] = dialogue_id
            scores["method"] = "baseline"
            baseline_eval_scores.append(scores)
            u = scores.get("therapist_skills", {}).get("understanding")
            print(f"  -> Done. Score (Understanding): {u}")

        # --- Emotion (text-only) ---
        if dialogue_id in emotion_dialogues:
            print(f"\nEvaluating EMOTION (text-only) Dialogue {dialogue_id}...")
            session_text = format_conversation_text(emotion_dialogues[dialogue_id], is_baseline=False)
            scores = evaluate_session(session_text)
            scores["dialogue_id"] = dialogue_id
            scores["method"] = "emotion_text_only"
            emotion_eval_scores.append(scores)
            u = scores.get("therapist_skills", {}).get("understanding")
            print(f"  -> Done. Score (Understanding): {u}")

        # --- Dissonance ---
        if dialogue_id in dissonance_dialogues:
            print(f"Evaluating DISSONANCE Dialogue {dialogue_id}...")
            session_text = format_conversation_text(dissonance_dialogues[dialogue_id], is_baseline=False)
            scores = evaluate_session(session_text)
            scores["dialogue_id"] = dialogue_id
            scores["method"] = "dissonance"
            dissonance_eval_scores.append(scores)
            u = scores.get("therapist_skills", {}).get("understanding")
            print(f"  -> Done. Score (Understanding): {u}")

    print("\n🎉 --- Evaluation Process Complete! ---")

--- Starting Evaluation Process (3 methods, 3 files) ---
Loaded 1 dialogue per method (baseline / emotion / dissonance).


Evaluating Dialogues:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating BASELINE Dialogue 1...
  -> Done. Score (Understanding): 5.0

Evaluating EMOTION (text-only) Dialogue 1...
  -> Done. Score (Understanding): 5.0
Evaluating DISSONANCE Dialogue 1...
  -> Done. Score (Understanding): 4.0

🎉 --- Evaluation Process Complete! ---


In [5]:
# Save Results

# Define output filenames (เรียงจากอ่อนไปเก่ง: baseline -> emotion -> dissonance)
BASELINE_MODEL_SCORE_FILE   = "ai_evaluation_results_BASELINE.jsonl"
EMOTION_MODEL_SCORE_FILE    = "ai_evaluation_results_EMOTION_TEXT_ONLY.jsonl"
DISSONANCE_MODEL_SCORE_FILE = "ai_evaluation_results_DISSONANCE.jsonl"

# --- Save Baseline Model Scores ---
try:
    with open(BASELINE_MODEL_SCORE_FILE, 'w', encoding='utf-8') as f:
        for entry in baseline_eval_scores:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    print(f"✅ Successfully saved 'Baseline' evaluation scores to '{BASELINE_MODEL_SCORE_FILE}'")
except Exception as e:
    print(f"❌ Error saving 'Baseline' scores: {e}")

# --- Save Emotion (text-only) Scores ---
try:
    with open(EMOTION_MODEL_SCORE_FILE, 'w', encoding='utf-8') as f:
        for entry in emotion_eval_scores:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    print(f"✅ Successfully saved 'Emotion (text-only)' evaluation scores to '{EMOTION_MODEL_SCORE_FILE}'")
except Exception as e:
    print(f"❌ Error saving 'Emotion (text-only)' scores: {e}")

# --- Save Dissonance Model Scores ---
try:
    with open(DISSONANCE_MODEL_SCORE_FILE, 'w', encoding='utf-8') as f:
        for entry in dissonance_eval_scores:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    print(f"✅ Successfully saved 'Dissonance' evaluation scores to '{DISSONANCE_MODEL_SCORE_FILE}'")
except Exception as e:
    print(f"❌ Error saving 'Dissonance' scores: {e}")

✅ Successfully saved 'Baseline' evaluation scores to 'ai_evaluation_results_BASELINE.jsonl'
✅ Successfully saved 'Emotion (text-only)' evaluation scores to 'ai_evaluation_results_EMOTION_TEXT_ONLY.jsonl'
✅ Successfully saved 'Dissonance' evaluation scores to 'ai_evaluation_results_DISSONANCE.jsonl'


In [6]:
import numpy as np

# --- 1) ฟังก์ชันคำนวณค่าเฉลี่ย ---

def calculate_averages(score_list):
    df = pd.json_normalize(score_list)

    if 'error' in df.columns:
        df = df[df['error'].isnull()]

    if df.empty:
        return {
            "understanding_avg": 0,
            "interpersonal_effectiveness_avg": 0,
            "affective_bond_avg": 0,
            "ctrs_collab_avg": 0,
            "ctrs_guided_avg": 0,
            "ctrs_focus_avg": 0,
            "ctrs_strategy_avg": 0,
            "count": 0,
        }

    averages = {
        "understanding_avg": df['therapist_skills.understanding'].mean(),
        "interpersonal_effectiveness_avg": df['therapist_skills.interpersonal_effectiveness'].mean(),
        "affective_bond_avg": df['client_alliance.affective_bond'].mean(),
        "ctrs_collab_avg": df['ctrs.collaboration'].mean(),
        "ctrs_guided_avg": df['ctrs.guided_discovery'].mean(),
        "ctrs_focus_avg": df['ctrs.focus'].mean(),
        "ctrs_strategy_avg": df['ctrs.strategy'].mean(),
        "count": len(df),
    }
    return averages

# --- 2) คำนวณค่าเฉลี่ยแยก 3 methods ---

baseline_avg   = calculate_averages(baseline_eval_scores)
emotion_avg    = calculate_averages(emotion_eval_scores)
dissonance_avg = calculate_averages(dissonance_eval_scores)

num_dialogues = 1  # ตอนนี้มีไฟล์ละ 1 dialogue ถ้าอนาคตเพิ่มค่อยเปลี่ยน

# --- 3) สร้างตารางสรุป เรียง Baseline -> Emotion -> Dissonance ---

summary_data = {
    "Metric": [
        "Understanding (0-6)", 
        "Interpersonal Effectiveness (0-6)", 
        "Affective Bond (1-5)",
        "CTRS Collaboration (0-6)",
        "CTRS Guided Discovery (0-6)",
        "CTRS Focus (0-6)",
        "CTRS Strategy (0-6)",
        "Successful Dialogues",
    ],
    "Baseline": [
        f"{baseline_avg['understanding_avg']:.2f}",
        f"{baseline_avg['interpersonal_effectiveness_avg']:.2f}",
        f"{baseline_avg['affective_bond_avg']:.2f}",
        f"{baseline_avg['ctrs_collab_avg']:.2f}",
        f"{baseline_avg['ctrs_guided_avg']:.2f}",
        f"{baseline_avg['ctrs_focus_avg']:.2f}",
        f"{baseline_avg['ctrs_strategy_avg']:.2f}",
        f"{baseline_avg['count']} / {num_dialogues}",
    ],
    "Emotion (Text-Only)": [
        f"{emotion_avg['understanding_avg']:.2f}",
        f"{emotion_avg['interpersonal_effectiveness_avg']:.2f}",
        f"{emotion_avg['affective_bond_avg']:.2f}",
        f"{emotion_avg['ctrs_collab_avg']:.2f}",
        f"{emotion_avg['ctrs_guided_avg']:.2f}",
        f"{emotion_avg['ctrs_focus_avg']:.2f}",
        f"{emotion_avg['ctrs_strategy_avg']:.2f}",
        f"{emotion_avg['count']} / {num_dialogues}",
    ],
    "Dissonance-Aware": [
        f"{dissonance_avg['understanding_avg']:.2f}",
        f"{dissonance_avg['interpersonal_effectiveness_avg']:.2f}",
        f"{dissonance_avg['affective_bond_avg']:.2f}",
        f"{dissonance_avg['ctrs_collab_avg']:.2f}",
        f"{dissonance_avg['ctrs_guided_avg']:.2f}",
        f"{dissonance_avg['ctrs_focus_avg']:.2f}",
        f"{dissonance_avg['ctrs_strategy_avg']:.2f}",
        f"{dissonance_avg['count']} / {num_dialogues}",
    ],
}

# --- 4) แสดงผล Reasoning และ Comparative Advantage แยกตาม Method ---

def print_qualitative_feedback(method_name, score_list):
    print(f"\n--- {method_name} Qualitative Feedback ---")
    for i, score in enumerate(score_list):
        # ดึงข้อมูลจาก dictionary (ถ้าไม่มีให้แสดง 'N/A')
        reasoning = score.get('reasoning', 'N/A')
        advantage = score.get('comparative_advantage', 'N/A')
        
        print(f"Dialogue {i+1}:")
        print(f"  > Reasoning: {reasoning}")
        print(f"  > Comparative Advantage: {advantage}")
    print("-" * 50)

summary_df = pd.DataFrame(summary_data)

print("Average Scores Across All Evaluated Dialogues:")
display(summary_df)

# เรียกใช้งาน summary reasoning
print_qualitative_feedback("Baseline", baseline_eval_scores)
print_qualitative_feedback("Emotion (Text-Only)", emotion_eval_scores)
print_qualitative_feedback("Dissonance-Aware", dissonance_eval_scores)

Average Scores Across All Evaluated Dialogues:


,Metric,Baseline,Emotion (Text-Only),Dissonance-Aware
0,Understanding (0-6),5.00,5.00,4.00
1,Interpersonal Effectiveness (0-6),5.00,5.00,5.00
2,Affective Bond (1-5),4.00,4.00,4.00
3,CTRS Collaboration (0-6),5.00,5.00,4.00
4,CTRS Guided Discovery (0-6),4.00,4.00,3.00
5,CTRS Focus (0-6),5.00,5.00,4.00
6,CTRS Strategy (0-6),4.00,4.00,3.00
7,Successful Dialogues,1 / 1,1 / 1,1 / 1



--- Baseline Qualitative Feedback ---
Dialogue 1:
  > Reasoning: The therapist demonstrated a strong understanding of the client's concerns, particularly in turns 2, 4, and 6, where they acknowledged the client's feelings of anxiety, guilt, and fear of burdening others. The therapist effectively maintained a supportive relationship throughout the session, as seen in their consistent validation of the client's feelings (e.g., turns 2, 4, 6, 8). However, the therapist could have delved deeper into the client's latent concerns, such as the underlying guilt and worthiness issues, which were hinted at but not fully explored. The therapist's approach to guided discovery was somewhat effective, as they encouraged the client to consider small steps towards connection (e.g., turns 10, 12), but they could have facilitated deeper exploration of the client's fears and anxieties. The focus was maintained on the client's primary concerns, and the strategy of suggesting writing as a first step was a